In [ ]:
!pip install crewai[google-genai]>=1.15

In [ ]:
!pip install requests

In [ ]:
import os
from getpass import getpass

import requests
from crewai import LLM, Agent, Crew, Process, Task
from crewai.tools import tool
from dotenv import load_dotenv

os.environ["CREWAI_TRACING_ENABLED"] = "false"

load_dotenv(".env")
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

llm = LLM(model="gemini/gemini-3.5-flash-lite", api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
@tool("Wikipedia Search")
def search_wikipedia(topic: str) -> str:
    """Search Wikipedia for a topic and return the opening of the best-matching article."""
    headers = {"User-Agent": "GenAI-course/1.0 (educational use)"}
    found = requests.get("https://en.wikipedia.org/w/api.php", headers=headers,
                         params={"action": "query", "list": "search", "srsearch": topic, "format": "json"}).json()
    results = found["query"]["search"]
    if len(results) == 0:
        return f"No Wikipedia article found for '{topic}'."
    title = results[0]["title"]
    page = requests.get("https://en.wikipedia.org/w/api.php", headers=headers,
                        params={"action": "query", "prop": "extracts", "exintro": 1, "explaintext": 1,
                                "titles": title, "format": "json"}).json()
    for page_id in page["query"]["pages"]:
        return f"{title}: " + page["query"]["pages"][page_id].get("extract", "")[:2500]
    return "No text found."

In [ ]:
researcher = Agent(role="Researcher", goal="Find accurate facts using Wikipedia.",
                   backstory="You only report what your sources say.", tools=[search_wikipedia], llm=llm)
writer = Agent(role="Writer", goal="Write a short, clear article from the research.",
               backstory="You write simply and only use the facts you're given.", llm=llm)
editor = Agent(role="Editor", goal="Polish the article and announce it to the whole team.",
               backstory="You fix mistakes, tighten the writing, and keep everyone informed.", llm=llm)

In [ ]:
## SHARED MEMORY
research_task = Task(
    description="Find 4 facts about {topic} using Wikipedia.",
    expected_output="4 bullet points of facts.",
    agent=researcher,
)

In [ ]:
writing_task = Task(
    description="Write a 120-word article about {topic} using the research.",
    expected_output="A 120-word article.",
    agent=writer,
    context=[research_task],
)

In [ ]:
editing_task = Task(
    description="Edit the article for clarity. Then add a final line that starts with 'ANNOUNCEMENT TO ALL TEAMS:' "
                "telling everyone the article is ready and what it's about.",
    expected_output="The edited article, then the announcement line.",
    agent=editor,
    context=[writing_task],
)

In [ ]:
crew=Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    max_rpm=4
)

In [ ]:
result = await crew.kickoff_async(inputs={"topic": "Generative AI"})

In [ ]:
for to in result.tasks_output:
  print("\n\n ========================== \n\n")
  print(to)

In [ ]:
project_manager = Agent(
    role="Project Manager",
    goal="Give me a high quality edited brief on the topic of interest",
    backstory="You run a high throughput, editorial team with Researcher, Writer, and Editor agents.",
    llm=llm,
)

In [ ]:
report_task = Task(
    description="Create a high quality researched article for me on the topic of interest, "
                  "bring in the experts as needed review validate and share the structured report",
    expected_output="A well-structured report in Markdown, about 300 words.",
)

In [ ]:
crew=Crew(
    agents=[researcher, writer, editor],
    tasks=[report_task],
    process=Process.hierarchical,
    manager_agent=project_manager,
    max_rpm=4,
    verbose=True
)

In [ ]:
result = await crew.kickoff_async(inputs={"topic": "Generative AI"})

In [ ]:
## The Three Cs

In [ ]:
# Communication

# 1:1, 1:n, n:n

# 1:1

In [ ]:
## Request Response
# Customer -> Support bot
# Customer's Request: Where is my order?
# Support Bot's Response: Your order has been shipped, should reach you by tomorrow

In [ ]:
# Pub Sub/Broadcast
# Server Gone Down
# One of the Agent - Publishes the message
# Topic that other agents have subscribed to and others are notified

In [ ]:
# Blackboard
## You have a shared memory,

Pattern -> Flow Type -> Style -> Examples -> Pros -> Cons
Request-Response -> 1:1 -> wait for an answer -> order -> Simple -> blocks
pub sub -> 1:n -> No wait -> Announcement -> Scalable -> No Feedback
Shared mem -> n:n ->no -> multicast -> Flexibility -> Conflicts

In [ ]:
# Dev, QA, Writer

In [ ]:
# Centralized
# Decentralized
# Rule Based
# Learning Based

In [ ]:
@tool("Customer support data")
def get_support_data(query: str) -> str:
    """Get last quarter's customer support summary: ticket counts, resolution times and feedback."""
    return ("Last quarter's support data:\n"
            "- 50 tickets: login issues (average 48 hours to resolve)\n"
            "- 30 tickets: billing errors (average 12 hours to resolve)\n"
            "- 20 tickets: feature requests (often closed without a fix)\n"
            "- Feedback: the password reset screen is confusing\n"
            "- Many calls about account verification")
print("Tool ready.")

In [ ]:
data_analyst = Agent(role="Customer support data analyst", goal="Find the biggest problems in the support data, with numbers.",
                     backstory="You read support data carefully and always quote the figures.", tools=[get_support_data], llm=llm)
process_optimizer = Agent(role="Process improvement specialist", goal="Turn the analysis into practical improvements.",
                          backstory="You suggest specific, realistic fixes, not vague advice.", llm=llm)
report_writer = Agent(role="Executive report writer", goal="Write a short, clear report for the company's COO.",
                      backstory="Your reports are brief, specific and easy to act on.", llm=llm)
print("Three workers ready.")

In [ ]:
analysis_task = Task(description="Analyse last quarter's customer support data and find the main problems.",
                     expected_output="The main problems, with ticket counts and resolution times.")
improvement_task = Task(description="Based on the analysis, suggest three concrete improvements to the support process.",
                        expected_output="Three improvements, each one or two sentences.", context=[analysis_task])
coo_report_task = Task(description="Write a short report for the COO: a summary, the problems, and the improvements.",
                       expected_output="A report of about 200 words.", context=[analysis_task, improvement_task])

manager_crew = Crew(agents=[data_analyst, process_optimizer, report_writer],
                    tasks=[analysis_task, improvement_task, coo_report_task],
                    process=Process.hierarchical, manager_llm=llm, max_rpm=4, verbose=True)

result = await manager_crew.kickoff_async()
print("\n=== FINAL REPORT ===\n" + result.raw)

In [ ]:
ai_researcher = Agent(role="AI research analyst", goal="Find well-documented facts about AI in software development, using Wikipedia.",
                      backstory="You never state a fact you haven't found in your research.", tools=[search_wikipedia], llm=llm)
analyst = Agent(role="Data analyst", goal="Pick the most promising idea from the research and explain why.",
                backstory="You turn raw research into a clear recommendation.", llm=llm)
summary_writer = Agent(role="Executive summary writer", goal="Write a short, polished summary for a business audience.",
                       backstory="You write clearly and only use facts you were given.", llm=llm)
print("Three pipeline stages ready.")

In [ ]:
ai_research_task = Task(description="Research three uses of AI in software development: large language models, automation and MLOps. Use Wikipedia.",
                        expected_output="A bullet list of three uses, two sentences each.", agent=ai_researcher)
choice_task = Task(description="From the research, choose the most promising use for a new software product, and explain why.",
                   expected_output="The chosen use, and three bullet-point reasons.", agent=analyst, context=[ai_research_task])
summary_task = Task(description="Write a one-page summary titled 'AI in Software Development': the chosen use, the reasons, a conclusion.",
                    expected_output="A summary of about 200 words.", agent=summary_writer, context=[choice_task])

pipeline_crew = Crew(agents=[ai_researcher, analyst, summary_writer], tasks=[ai_research_task, choice_task, summary_task],
                     process=Process.sequential, max_rpm=4)

result = await pipeline_crew.kickoff_async()
for task_output in result.tasks_output:
    print(f"=== {task_output.agent} ===\n{task_output.raw}\n")

In [ ]:
developer = Agent(role="Software developer", goal="Write clean, correct, well-commented Python code.",
                  backstory="You write careful code and welcome questions from teammates.", llm=llm, allow_delegation=True)
tester = Agent(role="QA engineer", goal="Find bugs and edge cases before users do.",
               backstory="You think about what could go wrong. If something is unclear, you ask the developer.", llm=llm, allow_delegation=True)
doc_writer = Agent(role="Technical writer", goal="Write clear, accurate documentation for other developers.",
                   backstory="You explain code simply, and you check details with the developer when unsure.", llm=llm, allow_delegation=True)
print("Three peers ready.")

In [ ]:
coding_task = Task(description="Write a Python function called calculate_average that takes a list of numbers and returns their average. "
                               "Handle an empty list sensibly. Add comments.",
                   expected_output="The complete Python function.", agent=developer)
testing_task = Task(description="Review calculate_average. Think through test cases: positive numbers, negative numbers, and an empty list. "
                                "Report any bugs you find.",
                    expected_output="The test cases, the expected result of each, and any bugs found.", agent=tester, context=[coding_task])
docs_task = Task(description="Write documentation for calculate_average: what it does, its parameter, what it returns, "
                             "what happens with an empty list, and two usage examples.",
                 expected_output="Short documentation in Markdown.", agent=doc_writer, context=[coding_task, testing_task])

peer_crew = Crew(agents=[developer, tester, doc_writer], tasks=[coding_task, testing_task, docs_task],
                 process=Process.sequential, max_rpm=4, verbose=True)

result = await peer_crew.kickoff_async()
print("\n=== FINAL DOCUMENTATION ===\n" + result.raw)

In [ ]:
# Research team
lead_researcher = Agent(role="Lead researcher", goal="Find well-documented information about robotics trends, using Wikipedia.",
                        backstory="You research carefully and only report what your sources say.", tools=[search_wikipedia], llm=llm)
market_analyst = Agent(role="Market analyst", goal="Turn research into clear insights about market impact.",
                       backstory="You spot patterns and explain why they matter for businesses.", llm=llm)
# Marketing team
strategist = Agent(role="Content strategist", goal="Turn findings into an engaging summary for executives.",
                   backstory="You write clear executive summaries from the facts you're given.", llm=llm)
# The manager: no tools
project_manager = Agent(role="Project manager", goal="Deliver a high-quality report by giving each part of the work to the right specialist.",
                        backstory="You run research and marketing teams, and you know who is best for each job.", llm=llm)
print("Two teams and a manager ready.")

In [ ]:
robotics_task = Task(description="Create a report on three important robotics trends: (1) a short overview of each, "
                                 "(2) its likely market impact, (3) a summary for executives.",
                     expected_output="A well-structured report in Markdown, about 300 words.")   # no agent=

federated_crew = Crew(agents=[lead_researcher, market_analyst, strategist],   # the manager is NOT in this list
                      tasks=[robotics_task], process=Process.hierarchical, manager_agent=project_manager,
                      max_rpm=4, verbose=True)

result = await federated_crew.kickoff_async()
print("\n=== FINAL REPORT ===\n" + result.raw)

In [ ]:
result.raw